In [32]:
# ==========================================================
# Notebook 02
# Dataset B
# Country Feature Engineering
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import networkx as nx

In [3]:
from google.colab import files

uploaded = files.upload()

Saving Dataset_A_Clean_Country_Corrected.csv to Dataset_A_Clean_Country_Corrected.csv


In [4]:
country_df = pd.read_csv(
    "/content/Dataset_A_Clean_Country_Corrected.csv",
    encoding="utf-8-sig"
)

In [5]:
date_columns = [
    "Date of First Case(s)",
    "Date of First Death(s)"
]

for col in date_columns:

    country_df[col] = pd.to_datetime(
        country_df[col],
        errors="coerce"
    )

In [6]:
global_first_case = country_df[
    "Date of First Case(s)"
].min()

print(global_first_case)

2019-11-17 00:00:00


In [7]:
country_df["Days_Since_Global_First_Case"] = (

    country_df["Date of First Case(s)"]

    - global_first_case

).dt.days

In [8]:
country_df["First_Case_Month"] = (
    country_df["Date of First Case(s)"]
    .dt.month
)

In [9]:
country_df["First_Case_Month_Name"] = (
    country_df["Date of First Case(s)"]
    .dt.month_name()
)

In [10]:
country_df["First_Case_Week"] = (
    country_df["Date of First Case(s)"]
    .dt.isocalendar()
    .week
)

In [11]:
country_df["First_Case_DayOfYear"] = (
    country_df["Date of First Case(s)"]
    .dt.dayofyear
)

In [12]:
country_df["First_Case_Quarter"] = (
    country_df["Date of First Case(s)"]
    .dt.quarter
)

In [13]:
country_df["First_Case_Year"] = (
    country_df["Date of First Case(s)"]
    .dt.year
)

In [14]:
country_df["Has_Travel_Origin"] = (
    country_df[
        "Last Visited Country(s) of Confirmed Case(s)"
    ]
    .notna()
)

In [15]:
country_df["Origin_Count"] = (

    country_df[
        "Last Visited Country(s) of Confirmed Case(s)"
    ]

    .fillna("")

    .apply(

        lambda x:

        len([i for i in x.split(",") if i.strip() != ""])

    )

)

In [16]:
country_df["Multiple_Origins"] = (

    country_df["Origin_Count"] > 1

)

In [17]:
country_df["Origin_List"] = (

    country_df[
        "Last Visited Country(s) of Confirmed Case(s)"
    ]

    .fillna("")

    .apply(

        lambda x: [

            i.strip()

            for i in x.split(",")

            if i.strip() != ""

        ]

    )

)

In [18]:
country_df = country_df.explode("Origin_List")

In [19]:
country_df = country_df.rename(

    columns={

        "Origin_List": "Origin"

    }

)

In [20]:
country_df["Origin"] = (

    country_df["Origin"]

    .replace("", pd.NA)

)

In [21]:
print(country_df[["Country", "Origin"]].head(20))

print()

print("Rows:", len(country_df))

                 Country            Origin
0                  China  Starting Country
1                 France               NaN
2                  Nepal             China
3               Thailand             China
4                  Japan             China
5            South Korea             China
6          United States             China
7                 Taiwan             China
8              Singapore             China
9                Vietnam             China
10             Australia             China
11              Malaysia             China
12                Canada             China
13             Sri Lanka             China
14               Germany             China
15              Cambodia             China
16  United Arab Emirates             China
17           Philippines             China
18                 India             China
19                 Italy             China

Rows: 215


In [22]:
country_df["Death_Recorded"] = (
    country_df[
        "Date of First Death(s)"
    ]
    .notna()
)

In [23]:
country_df["Days_To_First_Death"] = (

    country_df[
        "Date of First Death(s)"
    ]

    -

    country_df[
        "Date of First Case(s)"
    ]

).dt.days

In [24]:
country_df["Multiple_First_Day_Cases"] = (

    country_df[
        "Confirmed Case(s) at First Day"
    ] > 1

)

In [25]:
country_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 215 entries, 0 to 185
Data columns (total 22 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   Continent                                     215 non-null    object        
 1   Country                                       215 non-null    object        
 2   Date of First Case(s)                         215 non-null    datetime64[ns]
 3   Last Visited Country(s) of Confirmed Case(s)  202 non-null    object        
 4   Confirmed Case(s) at First Day                215 non-null    int64         
 5   Age of First Case(s)                          139 non-null    object        
 6   Date of First Death(s)                        187 non-null    datetime64[ns]
 7   Age of First Death(s)                         151 non-null    object        
 8   Days_Since_Global_First_Case                  215 non-null    int64        

In [26]:
country_df.head()

,Continent,Country,Date of First Case(s),Last Visited Country(s) of Confirmed Case(s),Confirmed Case(s) at First Day,Age of First Case(s),Date of First Death(s),Age of First Death(s),Days_Since_Global_First_Case,First_Case_Month,...,First_Case_DayOfYear,First_Case_Quarter,First_Case_Year,Has_Travel_Origin,Origin_Count,Multiple_Origins,Origin,Death_Recorded,Days_To_First_Death,Multiple_First_Day_Cases
0,Asia,China,2019-11-17,Starting Country,1,55,NaT,NaN,0,11,...,321,4,2019,True,1,False,Starting Country,False,NaN,False
1,Europe,France,2019-12-27,NaN,1,43,2020-02-15,80,40,12,...,361,4,2019,False,0,False,NaN,True,50.0,False
2,Asia,Nepal,2020-01-13,China,1,32,2020-05-16,29,57,1,...,13,1,2020,True,1,False,China,True,124.0,False
3,Asia,Thailand,2020-01-13,China,1,61,2020-03-01,35,57,1,...,13,1,2020,True,1,False,China,True,48.0,False
4,Asia,Japan,2020-01-16,China,1,30+,2020-02-13,80+,60,1,...,16,1,2020,True,1,False,China,True,28.0,False


In [27]:
country_df.describe(include="all")

,Continent,Country,Date of First Case(s),Last Visited Country(s) of Confirmed Case(s),Confirmed Case(s) at First Day,Age of First Case(s),Date of First Death(s),Age of First Death(s),Days_Since_Global_First_Case,First_Case_Month,...,First_Case_DayOfYear,First_Case_Quarter,First_Case_Year,Has_Travel_Origin,Origin_Count,Multiple_Origins,Origin,Death_Recorded,Days_To_First_Death,Multiple_First_Day_Cases
count,215,215,215,202,215.000000,139,187,151,215.000000,215.000000,...,215.000000,215.000000,215.000000,215,215.000000,215,202,215,187.000000,215
unique,6,186,NaN,53,NaN,69,NaN,63,NaN,NaN,...,NaN,NaN,NaN,2,NaN,2,45,2,NaN,2
top,Africa,Togo,NaN,Italy,NaN,38,NaN,60,NaN,NaN,...,NaN,NaN,NaN,True,NaN,False,Italy,True,NaN,False
freq,67,4,NaN,43,NaN,6,NaN,10,NaN,NaN,...,NaN,NaN,NaN,202,NaN,162,50,187,NaN,149
mean,NaN,NaN,2020-03-05 00:00:00,NaN,1.544186,NaN,2020-03-23 16:33:22.139037440,NaN,109.000000,2.725581,...,68.395349,1.065116,2019.990698,NaN,1.265116,NaN,NaN,NaN,19.721925,NaN
min,NaN,NaN,2019-11-17 00:00:00,NaN,1.000000,NaN,2020-02-01 00:00:00,NaN,0.000000,1.000000,...,13.000000,1.000000,2019.000000,NaN,0.000000,NaN,NaN,NaN,-1.000000,NaN
25%,NaN,NaN,2020-02-27 12:00:00,NaN,1.000000,NaN,2020-03-15 12:00:00,NaN,102.500000,2.000000,...,59.000000,1.000000,2020.000000,NaN,1.000000,NaN,NaN,NaN,10.000000,NaN
50%,NaN,NaN,2020-03-07 00:00:00,NaN,1.000000,NaN,2020-03-23 00:00:00,NaN,111.000000,3.000000,...,67.000000,1.000000,2020.000000,NaN,1.000000,NaN,NaN,NaN,17.000000,NaN
75%,NaN,NaN,2020-03-16 00:00:00,NaN,2.000000,NaN,2020-03-31 00:00:00,NaN,120.000000,3.000000,...,77.000000,1.000000,2020.000000,NaN,1.000000,NaN,NaN,NaN,24.500000,NaN
max,NaN,NaN,2020-05-13 00:00:00,NaN,15.000000,NaN,2020-05-16 00:00:00,NaN,178.000000,12.000000,...,361.000000,4.000000,2020.000000,NaN,4.000000,NaN,NaN,NaN,124.000000,NaN


In [28]:
output_file = "Dataset_B_Country_Features.csv"

country_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Dataset B saved!")
print(country_df.shape)

✅ Dataset B saved!
(215, 22)


In [29]:
from google.colab import files

files.download("Dataset_B_Country_Features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
print(country_df["Country"].nunique())
print(country_df["Origin"].nunique())

G = nx.from_pandas_edgelist(
    country_df.dropna(subset=["Origin"]),
    source="Origin",
    target="Country",
    create_using=nx.DiGraph()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

186
45
Nodes: 178
Edges: 202
